# Assignment 5 — Annual Report Fanciness (Google Colab GPU)

This notebook runs the **same version-controlled classifier in GitHub**, using a Colab session instead of your laptop.

**Before running:** In Colab choose **Runtime → Change runtime type → T4 GPU** (or another available GPU). Free GPU access is not guaranteed and Colab sessions may end. Don't upload SDU's licensed Orbis financial export here; this notebook needs only publicly available annual-report PDFs and a local company manifest.

**Important:** First test just **one verified PDF**. A link returning 403 in Windows may also fail in Colab. Upload the PDF manually if necessary. Keep report URLs and years verified against official company sources. No scores or findings are fabricated.

In [ ]:
import subprocess
gpu=subprocess.run(["nvidia-smi"],capture_output=True,text=True)
if gpu.returncode != 0:
    raise RuntimeError("No NVIDIA GPU detected. In Colab choose Runtime > Change runtime type > T4 GPU, if available.")
print(gpu.stdout[:3000])

## 1. Get the current scripts from GitHub

In [ ]:
import os, pathlib, subprocess
ROOT=pathlib.Path("/content/annual-report-fanciness-analysis")
if not ROOT.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/MVEST22/annual-report-fanciness-analysis.git",str(ROOT)],check=True)
else:
    subprocess.run(["git","-C",str(ROOT),"pull","--ff-only"],check=True)
os.chdir(ROOT)
subprocess.run(["python","-m","pip","-q","install","-r","requirements.txt"],check=True)
(ROOT/"reports").mkdir(exist_ok=True)
print("Project ready:",ROOT)

## 2. Install and start Ollama inside this Colab session

This runs **Gemma 3 locally within the Colab VM**; it does not require a paid API key. Model downloads consume session time and storage. The model still needs a suitable GPU to perform well.

In [ ]:
import os, subprocess, time, requests, pathlib
# Official Ollama Linux installer. Review the installation script if your security policy requires it.
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",shell=True,check=True,executable="/bin/bash")
log=open("/content/ollama-server.log","a")
server=subprocess.Popen(["ollama","serve"],stdout=log,stderr=subprocess.STDOUT,env=os.environ.copy())
for i in range(40):
    try:
        if requests.get("http://127.0.0.1:11434/api/tags",timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not become ready; inspect /content/ollama-server.log")
subprocess.run(["ollama","pull","gemma3:4b"],check=True)
print("Ollama model ready")

## 3. Upload your report manifest

Upload a CSV containing `company_name,bvd_id,report_url,report_year`, such as `report_manifest.csv`. The uploaded file is stored only in this temporary session. **Do not upload licensed Orbis profit data.**

In [ ]:
from google.colab import files
uploaded=files.upload()
csv_names=[n for n in uploaded if n.lower().endswith(".csv")]
if len(csv_names)!=1: raise ValueError("Upload exactly one manifest CSV in this step.")
manifest_path=ROOT/"report_manifest.csv"
manifest_path.write_bytes(uploaded[csv_names[0]])
import csv
with manifest_path.open(newline="",encoding="utf-8-sig") as f:
    rows=list(csv.DictReader(f))
assert rows and {"company_name","bvd_id","report_url","report_year"}.issubset(rows[0]), "Manifest is missing required columns."
print("Manifest records:",len(rows))
for i,r in enumerate(rows[:10]): print(i+1,r["company_name"],r["bvd_id"])

## 4. Upload one manually verified annual-report PDF

Choose a report from the manifest above and enter its **row number**. Upload the PDF from your own device; it will be stored under `reports/<bvd_id>.pdf`. This avoids 403-blocked direct links. **Check that the PDF belongs to that company and is its latest annual report.**

In [ ]:
from google.colab import files
selection=int(input("Which manifest row number (1-based) is this PDF for? "))
if not 1 <= selection <= len(rows): raise ValueError("Row number outside manifest.")
company=rows[selection-1]
selected_id=company["bvd_id"]
print("Selected:",company["company_name"],selected_id)
upload_pdf=files.upload()
pdf_files=[n for n in upload_pdf if n.lower().endswith(".pdf")]
if len(pdf_files)!=1: raise ValueError("Upload exactly one PDF.")
import re
safe_id=re.sub(r"[^a-zA-Z0-9_-]","_",selected_id)
target=ROOT/"reports"/(safe_id+".pdf")
pdfdata=upload_pdf[pdf_files[0]]
if not pdfdata.startswith(b"%PDF"): raise ValueError("The uploaded file does not appear to be a PDF.")
target.write_bytes(pdfdata)
with (ROOT/"test_manifest.csv").open("w",newline="",encoding="utf-8") as f:
    writer=csv.DictWriter(f,fieldnames=["company_name","bvd_id","report_url","report_year"])
    writer.writeheader();writer.writerow({k:company.get(k,"") for k in writer.fieldnames})
print("Ready to classify:",target.name,"(",len(pdfdata),"bytes)")

## 5. Run the one-report test

The classifier asks Gemma 3 to define 'fancy' and shows the definition before scoring. **Read and approve the definition.** This cell sends `YES` to the CLI; only run it if you have already reviewed and approved the same model/rubric in the prior local test. Otherwise, run the model-definition check separately first (next cell). The final five-dimension scoring uses the regular two-page method, not the single-cover pilot.

In [ ]:
from pathlib import Path
import requests
from IPython.display import display, Markdown
from classify_reports import RUBRIC
response=requests.post("http://127.0.0.1:11434/api/chat",
 json={"model":"gemma3:4b","stream":False,"messages":[{"role":"user","content":"Define your visual fanciness scale from 1 to 5 for corporate annual reports, without using profit or firm size. Compare it with: "+RUBRIC}],"options":{"temperature":0,"num_predict":450}},timeout=240)
response.raise_for_status()
definition=response.json()["message"]["content"]
print(definition)
(ROOT/"fanciness_definition.txt").write_text(definition+"\n\nResearcher rubric:\n"+RUBRIC,encoding="utf-8")
approval=input("Having reviewed the definition above, type YES to approve: ")
if approval.strip()!="YES": raise RuntimeError("Not approved; stop before classification.")
print("Definition approved for this session.")

In [ ]:
# Run one selected report with the previously approved definition.
import subprocess, sys
result=subprocess.run([sys.executable,"classify_reports.py","--manifest","test_manifest.csv",
                       "--limit","1","--reuse-definition"],input="YES\n",text=True,
                      cwd=str(ROOT),timeout=900)
if result.returncode: raise RuntimeError("Classifier returned an error.")
import pandas as pd
scores=pd.read_csv(ROOT/"scores.csv")
display(scores[["company_name","fanciness","weighted_score","status","rationale"]])
if not (scores["status"]=="scored").all():
    print("Not successfully scored: inspect the status and PDF contents before proceeding.")

## 6. Download the test results before leaving Colab

Colab VM files are temporary. Download `scores.csv` and `fanciness_definition.txt` to your computer, or deliberately save approved results to your own secure storage. **Do not overwrite your previous score files without backing them up.**

In [ ]:
from google.colab import files
for name in ["scores.csv","fanciness_definition.txt"]:
    path=ROOT/name
    if path.exists(): files.download(str(path))
print("Keep the downloaded files. The public GitHub repository stores the code, not the private Orbis dataset.")

## After the pilot

For larger-scale classification, first collect and validate the PDF manifest for each company. Upload report PDFs or a ZIP containing a `reports/` folder into the Colab session, then use the existing `classify_reports.py` script (without `--fast`) on the verified complete manifest. Split work into batches and export results after each batch. Missing PDFs, year mismatches and blocked links must remain flagged—not assigned invented scores. The regression will be executed locally with your licensed Orbis profit figures using `regression.py`.